# Q3 · T4 첫 검증 · 30분 목표

기존과 같은 영상 전용 모델을 학습 16개·검증 4개 영상으로 짧게 검사합니다.
실제 행동 정답은 읽지 않습니다. 과거 32장 → 미래 32장 예측과 copy-last를 비교합니다.
최대 300회 업데이트, 학습 시간 상한 10분입니다. 설정 실행부터 20분을 예산으로 두고
마지막 검증·저장에 2분을 남깁니다. GPU 할당·로그인 지연은 별도로 확인해야 합니다.


## 1. T4 검증 설정
약 20MB의 선택 영상 묶음을 사용합니다. GPU 런타임에서 셀을 순서대로 실행하세요.
설정 셀을 실행하는 순간 시간 예산을 시작합니다. 이 노트북은 짧은 학습을 실행합니다.
검증 4개에 대한 결과이므로 전체 데이터의 성능을 대표하지는 않습니다.


In [ ]:
import time
REPO_URL = "https://github.com/seungjoolee24/krafton-q3-video-worldmodel.git"
GIT_REF = "main"
PROFILE = "t4_quick"
RUN_NAME = "video-only-t4-first-v1"
DRIVE_ROOT = "/content/drive/MyDrive/krafton-q3-video-worldmodel"
DATA_ZIP = DRIVE_ROOT + "/track3-kit-t4-quick.zip"
DATA_SHA256 = "c9dc3afc3691245366de08f8c0fd6665fd5fb410bb136999b2147cd717955882"
RESUME_CHECKPOINT = ""
MAX_STEPS = 300
BATCH_SIZE = 4
TRAIN_BUDGET_SECONDS = 600
TOTAL_BUDGET_MINUTES = 20
QUICK_STARTED = time.time()
QUICK_DEADLINE = QUICK_STARTED + TOTAL_BUDGET_MINUTES * 60
print("Validation deadline:", time.strftime("%H:%M:%S UTC", time.gmtime(QUICK_DEADLINE)))


## 2. 코드와 GPU 환경
현재 Git 커밋을 출력합니다. 재현할 때 `GIT_REF`에 이 SHA를 사용하세요.
Colab에 설치된 CUDA Torch를 유지하고 나머지 의존성만 설치합니다.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
version = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert version >= (2, 3), f"Torch >= 2.3 required, found {torch.__version__}"
assert PROFILE in {"t4_quick"} and BATCH_SIZE >= 1 and MAX_STEPS >= 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME, "RUN_NAME은 폴더 이름 하나로 지정하세요."
CODE = Path("/content/q3-video-code")
if not (CODE / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(CODE), "remote", "get-url", "origin"], text=True).strip()
    assert remote == REPO_URL, "Existing checkout has a different repository."
    dirty = subprocess.check_output(["git", "-C", str(CODE), "status", "--porcelain"], text=True)
    assert not dirty.strip(), "Source changes exist; preserve them before changing revision."
subprocess.run(["git", "-C", str(CODE), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CODE), "--no-deps"], check=True)
# Editable-install .pth files are read by new processes, not the running kernel.
SOURCE_PATH = str(CODE / "src")
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
import importlib
importlib.invalidate_caches()
import video_wam
assert Path(video_wam.__file__).resolve().is_relative_to(CODE.resolve())
os.chdir(CODE)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
print({"commit": COMMIT, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})
subprocess.run([sys.executable, "scripts/check_project.py"], check=True)


## 3. Drive 연결과 원본 데이터
Google Drive 접근은 본인 계정으로 승인합니다.
데이터 ZIP을 런타임 로컬 디스크로 복사한 뒤 압축을 풉니다.
학습 중 Drive에서 수천 개의 파일을 반복해서 읽지 않습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
source_zip = Path(DATA_ZIP)
LOCAL_DATA = Path("/content/q3-data")
LOCAL_DATA.mkdir(exist_ok=True)
local_zip = LOCAL_DATA / "track3-kit.zip"
from importlib.util import spec_from_file_location, module_from_spec
specification = spec_from_file_location("unpack_kit", CODE / "scripts/unpack_kit.py")
unpack_module = module_from_spec(specification)
specification.loader.exec_module(unpack_module)
unpack_module.stage_archive(source_zip, local_zip, DATA_SHA256)
KIT = unpack_module.unpack(local_zip, LOCAL_DATA / "unpacked")
CACHE = Path(f"/content/q3-cache-{PROFILE}")
LOCAL_RUN = Path(f"/content/q3-runs/{RUN_NAME}")
DRIVE_RUN = DRIVE / "runs" / RUN_NAME
config = json.loads((CODE / "configs" / f"{PROFILE}.json").read_text())
config["batch_size"] = BATCH_SIZE
if MAX_STEPS:
    config["total_steps"] = MAX_STEPS
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOCAL_RUN / "config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print({"kit": str(KIT), "cache": str(CACHE), "drive_results": str(DRIVE_RUN), "config": config})


## 4. 짧은 동작 검증
작은 합성 입력의 gradient·배치 독립성·미래 정보 차단·재개를 검사합니다.
GPU smoke는 두 번의 작은 합성 업데이트만 수행하며 학습 가중치를 저장하지 않습니다.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "-m", "video_wam.cli", "smoke", "--kit", str(KIT), "--device", "cuda"], check=True)


## 5. 선택 영상 특징 캐시
원본 MP4 20개를 고정 인코더로 변환합니다. 캐시는 약 0.35GB입니다.
실제 액션 파일은 데이터 묶음에 포함되어 있지 않습니다.


In [ ]:
subprocess.run([sys.executable, "-m", "video_wam.cli", "prepare",
                "--kit", str(KIT), "--cache", str(CACHE), "--config", str(CONFIG_PATH),
                "--encode-batch", "64", "--device", "cuda"], check=True)
index = json.loads((CACHE / "index.json").read_text())
print({"train_episodes": len(index["train_ids"]), "dev_episodes": len(index["dev_ids"]),
       "actions_used": index["actions_used"], "cache_fingerprint": index["fingerprint"]})


## 6. 짧은 GPU 학습과 검증
학습 전 기준선을 먼저 저장하고, 최대 300회 또는 학습 시간 상한까지 업데이트합니다.
예측 길이는 1 → 4 → 8 → 16 → 32로 늘립니다. 종료 시점의 모델도 검증하고 저장합니다.
시간 상한은 업데이트 사이에서 적용하며 마지막 검증·저장 시간이 추가됩니다.


In [ ]:
remaining = QUICK_DEADLINE - time.time() - 120
assert remaining > 0, "Setup exhausted the time budget; check GPU/Drive connectivity."
training_budget = min(TRAIN_BUDGET_SECONDS, remaining)
print({"training_seconds_limit": training_budget, "max_updates": MAX_STEPS})
resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else DRIVE_RUN / "latest.pt"
command = [sys.executable, "-m", "video_wam.cli", "train", "--kit", str(KIT),
           "--cache", str(CACHE), "--config", str(CONFIG_PATH), "--out", str(LOCAL_RUN),
           "--persist", str(DRIVE_RUN), "--device", "cuda"]
if RESUME_CHECKPOINT:
    assert resume.is_file(), f"Resume checkpoint not found: {resume}"
if resume.is_file():
    command.extend(["--resume", str(resume)])
    print("Resume:", resume)
if MAX_STEPS:
    command.extend(["--max-steps", str(MAX_STEPS)])
command.extend(["--max-seconds", str(training_budget)])
subprocess.run(command, check=True)


## 7. 첫 검증 결과
학습 전후 RGB MSE, copy-last, 최종 손실과 비교 영상을 확인합니다.
아래 영상은 마지막 완료된 업데이트의 모델입니다. 최저 오차 모델은 별도의 best.pt입니다.
개선 여부는 결과로 판단합니다. 수치가 낮아도 실제 힘 복원이 검증된 것은 아닙니다.


In [ ]:
from IPython.display import display, Video
RESULT_ROOT = LOCAL_RUN if (LOCAL_RUN / "completion.json").exists() else DRIVE_RUN
completion = json.loads((RESULT_ROOT / "completion.json").read_text())
REPORT = LOCAL_RUN / "validation" / f"step_{completion['step']:06d}"
if not (REPORT / "metrics.json").exists():
    REPORT = DRIVE_RUN / "validation" / f"step_{completion['step']:06d}"
metrics = json.loads((REPORT / "metrics.json").read_text())
initial_path = RESULT_ROOT / "validation" / "step_000000" / "metrics.json"
initial = json.loads(initial_path.read_text()) if initial_path.exists() else None
loss_path = RESULT_ROOT / "training.jsonl"
logs = [json.loads(line) for line in loss_path.read_text().splitlines()] if loss_path.exists() else []
display({"completed_updates": completion["step"], "stop_reason": completion["reason"],
         "total_elapsed_minutes": (time.time() - QUICK_STARTED) / 60,
         "initial_rgb_mse": initial["mse"]["mean"] if initial else None,
         "final_rgb_mse": metrics["mse"]["mean"],
         "copy_last_rgb_mse": metrics["copy_last_mse"]["mean"],
         "final_horizon_errors": metrics["mse"],
         "last_training_log": logs[-1] if logs else None,
         "gpu_peak_memory_gb": completion["gpu_peak_memory_gb"],
         "actions_used": False, "saved_results": str(DRIVE_RUN)})
for video in sorted(REPORT.glob("*.mp4")):
    display(Video(str(video), embed=True, width=768))
print("Results saved:", DRIVE_RUN)
